In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

import joblib
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("../dataset/2013 dataset/creditcard.csv")

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["Class"].value_counts())

Dataset shape: (284807, 31)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64


In [3]:
print("Duplicate rows:", df.duplicated().sum())

df_clean = df.drop_duplicates().copy()

print("Original shape :", df.shape)
print("Cleaned shape  :", df_clean.shape)
print("\nClass distribution after cleaning:")
print(df_clean["Class"].value_counts())

Duplicate rows: 1081
Original shape : (284807, 31)
Cleaned shape  : (283726, 31)

Class distribution after cleaning:
Class
0    283253
1       473
Name: count, dtype: int64


In [4]:
X = df_clean.drop("Class", axis=1)
y = df_clean["Class"]

# First: 80% development, 20% final test
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Then: split development into 75% train, 25% validation
# Overall = 60% train, 20% validation, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Train :", X_train.shape, "| Fraud:", y_train.sum())
print("Val   :", X_val.shape,   "| Fraud:", y_val.sum())
print("Test  :", X_test.shape,  "| Fraud:", y_test.sum())

Train : (170235, 30) | Fraud: 284
Val   : (56745, 30) | Fraud: 94
Test  : (56746, 30) | Fraud: 95


In [5]:
scaler = StandardScaler()

X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

X_train["Amount"] = scaler.fit_transform(X_train[["Amount"]])
X_val["Amount"] = scaler.transform(X_val[["Amount"]])
X_test["Amount"] = scaler.transform(X_test[["Amount"]])

print("Amount scaling completed successfully! ✅")
print("Training Amount mean:", X_train["Amount"].mean())
print("Training Amount std :", X_train["Amount"].std())

Amount scaling completed successfully! ✅
Training Amount mean: 1.869904224281287e-17
Training Amount std : 1.0000029371292791


In [6]:
final_model = IsolationForest(
    n_estimators=200,
    contamination=0.005,
    random_state=42,
    n_jobs=-1
)

# Train only on legitimate transactions
X_train_normal = X_train[y_train == 0]

final_model.fit(X_train_normal)

print("Isolation Forest trained successfully! ✅")
print("Training samples:", len(X_train_normal))

Isolation Forest trained successfully! ✅
Training samples: 169951


In [7]:
val_decision_scores = final_model.decision_function(X_val)

# Higher value = more anomalous
val_anomaly_scores = -val_decision_scores

print("Validation scores generated! ✅")
print("Minimum anomaly score:", val_anomaly_scores.min())
print("Maximum anomaly score:", val_anomaly_scores.max())
print("Mean anomaly score   :", val_anomaly_scores.mean())

Validation scores generated! ✅
Minimum anomaly score: -0.2554998489898833
Maximum anomaly score: 0.13127873810322344
Mean anomaly score   : -0.1993702779275924


In [8]:
thresholds = np.linspace(
    val_anomaly_scores.min(),
    val_anomaly_scores.max(),
    200
)

results = []

for threshold in thresholds:
    y_val_pred = (val_anomaly_scores >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(results)

best_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("Best validation threshold:")
print(best_row)

Best validation threshold:
Threshold    0.036042
Precision    0.192308
Recall       0.265957
F1           0.223214
Name: 150, dtype: float64


In [9]:
selected_threshold = best_row["Threshold"]

print(f"Selected threshold: {selected_threshold:.6f}")
print("Threshold locked for final test evaluation. 🔒")

Selected threshold: 0.036042
Threshold locked for final test evaluation. 🔒


In [10]:
# Generate anomaly scores on the untouched test set
test_decision_scores = final_model.decision_function(X_test)
test_anomaly_scores = -test_decision_scores

# Apply the locked validation threshold
y_test_pred = (
    test_anomaly_scores >= selected_threshold
).astype(int)

# Metrics
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, zero_division=0)
recall = recall_score(y_test, y_test_pred, zero_division=0)
f1 = f1_score(y_test, y_test_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, test_anomaly_scores)
pr_auc = average_precision_score(y_test, test_anomaly_scores)

print("========== FINAL TEST RESULTS ==========")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

========== FINAL TEST RESULTS ==========
Accuracy  : 0.9968
Precision : 0.1818
Recall    : 0.2526
F1 Score  : 0.2115
ROC-AUC   : 0.9407
PR-AUC    : 0.0913

Confusion Matrix:
[[56543   108]
 [   71    24]]


In [11]:
# Save final trained model, scaler, and locked threshold

joblib.dump(final_model, "../models/final_isolation_forest.pkl")
joblib.dump(scaler, "../models/final_scaler.pkl")
joblib.dump(selected_threshold, "../models/fraud_threshold.pkl")

print("Final artifacts saved successfully! ✅")
print("\nSaved files:")
print("- final_isolation_forest.pkl")
print("- final_scaler.pkl")
print("- fraud_threshold.pkl")

Final artifacts saved successfully! ✅

Saved files:
- final_isolation_forest.pkl
- final_scaler.pkl
- fraud_threshold.pkl


In [12]:
import os

for file in [
    "../models/final_isolation_forest.pkl",
    "../models/final_scaler.pkl",
    "../models/fraud_threshold.pkl"
]:
    print(file, "→", "EXISTS ✅" if os.path.exists(file) else "MISSING ❌")

../models/final_isolation_forest.pkl → EXISTS ✅
../models/final_scaler.pkl → EXISTS ✅
../models/fraud_threshold.pkl → EXISTS ✅


In [14]:
import os

print("Folders inside dataset:")
for item in os.listdir("../dataset"):
    print(repr(item))

Folders inside dataset:
'2013 dataset'
'newer synthatic dataset'


In [15]:
dataset_path = "../dataset/newer synthatic dataset"

print("Files in newer dataset folder:")
for file in os.listdir(dataset_path):
    print("-", file)

Files in newer dataset folder:
- credit-card-fraud-xai-dataset-main


In [16]:
dataset_path = "../dataset/newer synthatic dataset/credit-card-fraud-xai-dataset-main"

print("Files/folders inside:")
for item in os.listdir(dataset_path):
    print("-", item)

Files/folders inside:
- credit_card_fraud_sample.csv
- dataset_summary.txt
- README.md


In [17]:
newer_file = (
    "../dataset/newer synthatic dataset/"
    "credit-card-fraud-xai-dataset-main/"
    "credit_card_fraud_sample.csv"
)

df_new = pd.read_csv(newer_file)

print("Shape:", df_new.shape)

print("\nColumns:")
print(df_new.columns.tolist())

print("\nFirst 5 rows:")
display(df_new.head())

print("\nData types:")
print(df_new.dtypes)

Shape: (4784, 38)

Columns:
['trans_date_trans_time', 'year', 'month', 'day', 'time', 'cc_num', 'card_network', 'user_id', 'card_id', 'payment_method', 'merchant', 'category', 'mcc', 'amount', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'region', 'lat', 'long', 'city_pop', 'job', 'income', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_foreign', 'errors', 'is_fraud', 'is_anomaly', 'days_since_last_transaction', 'transaction_amount_delta']

First 5 rows:


,trans_date_trans_time,year,month,day,time,cc_num,card_network,user_id,card_id,payment_method,...,trans_num,unix_time,merch_lat,merch_long,is_foreign,errors,is_fraud,is_anomaly,days_since_last_transaction,transaction_amount_delta
0,08-01-2023 12:53,2023,1,8,12:53,4027201288577493,Visa,21393,213931,Swipe Transaction,...,326ccee575f8dba3b98b70360f4e688d,1673162622,-15.011519,-44.876280,0,NaN,1,0,46.009062,326.43
1,08-05-2022 06:49,2022,5,8,06:49,4730618204104309,Visa,38730,387303,Swipe Transaction,...,b2d9141204ac1d81504bf03c41620798,1651972785,-47.087221,-58.496202,1,NaN,1,0,6.703924,721.25
2,04-11-2020 19:03,2020,11,4,19:03,5355122363204271,Mastercard,30802,308022,Swipe Transaction,...,eb9c23899d076d0eee869e0bddcd8dbe,1604496787,-9.407178,76.744171,1,NaN,1,0,17.994525,-81.22
3,16-01-2021 13:12,2021,1,16,13:12,2388722261365350,Mastercard,39015,390150,Online Transaction,...,13652df33675b51abe7575dc3aa13610,1610782971,18.079362,-96.521277,1,Connection Timeout,1,0,53.787859,18.50
4,05-09-2022 19:00,2022,9,5,19:00,4911923995512876,Visa,29396,293962,Swipe Transaction,...,0763a44525175ea8f335ee77bd5138bd,1662384645,-37.810058,-56.236248,1,Card Limit Exceeded,1,0,15.475382,174.34



Data types:
trans_date_trans_time              str
year                             int64
month                            int64
day                              int64
time                               str
cc_num                           int64
card_network                       str
user_id                          int64
card_id                          int64
payment_method                     str
merchant                           str
category                           str
mcc                              int64
amount                         float64
first                              str
last                               str
gender                             str
street                             str
city                               str
state                              str
zip                                str
region                             str
lat                            float64
long                           float64
city_pop                         int64
job         

In [18]:
import os
import pandas as pd

# ============================================================
# 1. Locate the newer dataset
# ============================================================

newer_file = (
    "../dataset/newer synthatic dataset/"
    "credit-card-fraud-xai-dataset-main/"
    "credit_card_fraud_sample.csv"
)

# ============================================================
# 2. Load dataset
# ============================================================

df_new = pd.read_csv(newer_file)

# ============================================================
# 3. Basic information
# ============================================================

print("=" * 60)
print("NEWER DATASET INSPECTION")
print("=" * 60)

print("\nShape:")
print(df_new.shape)

print("\nColumns:")
for i, col in enumerate(df_new.columns, 1):
    print(f"{i:2}. {col}")

print("\nData Types:")
print(df_new.dtypes)

# ============================================================
# 4. Missing values
# ============================================================

print("\nMissing Values:")
missing = df_new.isnull().sum()
print(missing[missing > 0])

if missing.sum() == 0:
    print("No missing values ✅")

# ============================================================
# 5. Duplicate rows
# ============================================================

print("\nDuplicate Rows:")
print(df_new.duplicated().sum())

# ============================================================
# 6. First 5 rows
# ============================================================

print("\nFirst 5 Rows:")
display(df_new.head())

# ============================================================
# 7. Last 5 rows
# ============================================================

print("\nLast 5 Rows:")
display(df_new.tail())

# ============================================================
# 8. Numerical summary
# ============================================================

print("\nNumerical Summary:")
display(df_new.describe(include="all").T)

# ============================================================
# 9. Possible target/class columns
# ============================================================

print("\nPotential Target Columns:")

possible_targets = [
    col for col in df_new.columns
    if any(word in col.lower()
           for word in ["class", "fraud", "is_fraud", "label", "target"])
]

if possible_targets:
    print(possible_targets)
else:
    print("No obvious target column found.")

# ============================================================
# 10. Class distribution for possible target columns
# ============================================================

for col in possible_targets:
    print(f"\nValue counts for '{col}':")
    print(df_new[col].value_counts(dropna=False))

NEWER DATASET INSPECTION

Shape:
(4784, 38)

Columns:
 1. trans_date_trans_time
 2. year
 3. month
 4. day
 5. time
 6. cc_num
 7. card_network
 8. user_id
 9. card_id
10. payment_method
11. merchant
12. category
13. mcc
14. amount
15. first
16. last
17. gender
18. street
19. city
20. state
21. zip
22. region
23. lat
24. long
25. city_pop
26. job
27. income
28. dob
29. trans_num
30. unix_time
31. merch_lat
32. merch_long
33. is_foreign
34. errors
35. is_fraud
36. is_anomaly
37. days_since_last_transaction
38. transaction_amount_delta

Data Types:
trans_date_trans_time              str
year                             int64
month                            int64
day                              int64
time                               str
cc_num                           int64
card_network                       str
user_id                          int64
card_id                          int64
payment_method                     str
merchant                           str
category          

,trans_date_trans_time,year,month,day,time,cc_num,card_network,user_id,card_id,payment_method,...,trans_num,unix_time,merch_lat,merch_long,is_foreign,errors,is_fraud,is_anomaly,days_since_last_transaction,transaction_amount_delta
0,08-01-2023 12:53,2023,1,8,12:53,4027201288577493,Visa,21393,213931,Swipe Transaction,...,326ccee575f8dba3b98b70360f4e688d,1673162622,-15.011519,-44.876280,0,NaN,1,0,46.009062,326.43
1,08-05-2022 06:49,2022,5,8,06:49,4730618204104309,Visa,38730,387303,Swipe Transaction,...,b2d9141204ac1d81504bf03c41620798,1651972785,-47.087221,-58.496202,1,NaN,1,0,6.703924,721.25
2,04-11-2020 19:03,2020,11,4,19:03,5355122363204271,Mastercard,30802,308022,Swipe Transaction,...,eb9c23899d076d0eee869e0bddcd8dbe,1604496787,-9.407178,76.744171,1,NaN,1,0,17.994525,-81.22
3,16-01-2021 13:12,2021,1,16,13:12,2388722261365350,Mastercard,39015,390150,Online Transaction,...,13652df33675b51abe7575dc3aa13610,1610782971,18.079362,-96.521277,1,Connection Timeout,1,0,53.787859,18.50
4,05-09-2022 19:00,2022,9,5,19:00,4911923995512876,Visa,29396,293962,Swipe Transaction,...,0763a44525175ea8f335ee77bd5138bd,1662384645,-37.810058,-56.236248,1,Card Limit Exceeded,1,0,15.475382,174.34



Last 5 Rows:


,trans_date_trans_time,year,month,day,time,cc_num,card_network,user_id,card_id,payment_method,...,trans_num,unix_time,merch_lat,merch_long,is_foreign,errors,is_fraud,is_anomaly,days_since_last_transaction,transaction_amount_delta
4779,07-05-2020 16:30,2020,5,7,16:30,345516432891469,American Express,14860,148601,Contactless Payment,...,e5b41167d906faac89ee22ccd7d5076c,1588849240,-11.523881,141.001833,0,NaN,0,0,0.694329,-29.94
4780,09-01-2020 09:38,2020,1,9,09:38,4699799500420608,Visa,67267,672670,Chip Transaction,...,78c4cee9792acb3c37e377313b05ebcc,1578542893,19.335581,150.094054,0,NaN,0,0,0.534178,78.31
4781,06-02-2021 12:45,2021,2,6,12:45,2221953062515265,Mastercard,12968,129680,Chip Transaction,...,b286128b028596abcf5b582bfedbc632,1612595703,54.469541,17.057397,0,NaN,0,0,22.068819,-31.04
4782,03-03-2023 10:48,2023,3,3,10:48,2226320684468687,Mastercard,39122,391221,Swipe Transaction,...,2d57daded8cde9670523f59868b64bc4,1677820729,-24.575601,-104.669904,0,NaN,0,0,124.017824,-49.62
4783,11-09-2022 13:34,2022,9,11,13:34,2227861099269290,Mastercard,62046,620461,Chip Transaction,...,ca95681b1332c1d8d8241efd73202d90,1662883481,38.037975,-1.530631,0,NaN,0,0,23.770463,62.42



Numerical Summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
trans_date_trans_time,4784,4775,29-03-2023 13:39,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,4784.0,NaN,NaN,NaN,2021.285953,1.022721,2020.0,2020.0,2021.0,2022.0,2023.0
month,4784.0,NaN,NaN,NaN,6.119774,3.388694,1.0,3.0,6.0,9.0,12.0
day,4784.0,NaN,NaN,NaN,15.691054,8.797384,1.0,8.0,16.0,23.0,31.0
time,4784,1035,20:27,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cc_num,4784.0,NaN,NaN,NaN,3545239348002878.5,1802775020017080.5,340025796032818.0,2237365658242077.75,4132458940724228.5,4774623083379766.0,6596317820327578.0
card_network,4784,5,Visa,1938,NaN,NaN,NaN,NaN,NaN,NaN,NaN
user_id,4784.0,NaN,NaN,NaN,34481.075669,20111.539217,7.0,17032.75,33973.0,51759.0,69976.0
card_id,4784.0,NaN,NaN,NaN,344811.313336,201115.395172,70.0,170327.5,339730.0,517590.25,699760.0
payment_method,4784,4,Chip Transaction,2245,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Potential Target Columns:
['is_fraud']

Value counts for 'is_fraud':
is_fraud
0    4000
1     784
Name: count, dtype: int64


In [19]:
# ============================================================
# COMPLETE NEWER DATASET SCHEMA CHECK
# ============================================================

print("=" * 70)
print("ALL COLUMNS")
print("=" * 70)

for i, col in enumerate(df_new.columns, 1):
    print(f"{i:2}. {col}")

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)

print(df_new.dtypes.to_string())

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing = df_new.isnull().sum()
print(missing[missing > 0])

if missing.sum() == 0:
    print("No missing values ✅")

print("\n" + "=" * 70)
print("POSSIBLE TARGET / FRAUD COLUMNS")
print("=" * 70)

possible_targets = [
    col for col in df_new.columns
    if any(word in col.lower()
           for word in ["fraud", "class", "label", "target"])
]

print(possible_targets)

for col in possible_targets:
    print(f"\n--- {col} ---")
    print(df_new[col].value_counts(dropna=False))

print("\n" + "=" * 70)
print("NUMERIC COLUMNS")
print("=" * 70)

print(df_new.select_dtypes(include="number").columns.tolist())

print("\n" + "=" * 70)
print("CATEGORICAL / TEXT COLUMNS")
print("=" * 70)

print(df_new.select_dtypes(include=["object", "category"]).columns.tolist())

ALL COLUMNS
 1. trans_date_trans_time
 2. year
 3. month
 4. day
 5. time
 6. cc_num
 7. card_network
 8. user_id
 9. card_id
10. payment_method
11. merchant
12. category
13. mcc
14. amount
15. first
16. last
17. gender
18. street
19. city
20. state
21. zip
22. region
23. lat
24. long
25. city_pop
26. job
27. income
28. dob
29. trans_num
30. unix_time
31. merch_lat
32. merch_long
33. is_foreign
34. errors
35. is_fraud
36. is_anomaly
37. days_since_last_transaction
38. transaction_amount_delta

DATA TYPES
trans_date_trans_time              str
year                             int64
month                            int64
day                              int64
time                               str
cc_num                           int64
card_network                       str
user_id                          int64
card_id                          int64
payment_method                     str
merchant                           str
category                           str
mcc                   

C:\Users\singh\AppData\Local\Temp\ipykernel_26708\4045900320.py:54: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df_new.select_dtypes(include=["object", "category"]).columns.tolist())


In [20]:
# ============================================================
# FINAL SCHEMA CHECK — NEWER DATASET
# ============================================================

print("=" * 70)
print("LAST 20 COLUMNS")
print("=" * 70)

for i, col in enumerate(df_new.columns[-20:], len(df_new.columns) - 19):
    print(f"{i:2}. {col}")

print("\n" + "=" * 70)
print("POSSIBLE TARGET COLUMNS")
print("=" * 70)

possible_targets = [
    col for col in df_new.columns
    if any(word in col.lower()
           for word in ["fraud", "class", "is_fraud", "label", "target"])
]

print(possible_targets)

for col in possible_targets:
    print(f"\nValue counts for '{col}':")
    print(df_new[col].value_counts(dropna=False))

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print("Rows:", len(df_new))
print("Columns:", len(df_new.columns))
print("Duplicates:", df_new.duplicated().sum())
print("Missing values:", df_new.isna().sum().sum())

LAST 20 COLUMNS
19. city
20. state
21. zip
22. region
23. lat
24. long
25. city_pop
26. job
27. income
28. dob
29. trans_num
30. unix_time
31. merch_lat
32. merch_long
33. is_foreign
34. errors
35. is_fraud
36. is_anomaly
37. days_since_last_transaction
38. transaction_amount_delta

POSSIBLE TARGET COLUMNS
['is_fraud']

Value counts for 'is_fraud':
is_fraud
0    4000
1     784
Name: count, dtype: int64

DATASET SUMMARY
Rows: 4784
Columns: 38
Duplicates: 0
Missing values: 4621


In [21]:
# ============================================================
# FINAL TARGET + MISSING-VALUE ANALYSIS
# ============================================================

print("=" * 70)
print("TARGET DISTRIBUTIONS")
print("=" * 70)

for col in ["is_fraud", "is_anomaly"]:
    print(f"\n--- {col} ---")
    print("Data type:", df_new[col].dtype)
    print("Missing:", df_new[col].isna().sum())
    print(df_new[col].value_counts(dropna=False))

print("\n" + "=" * 70)
print("MISSING VALUES BY COLUMN")
print("=" * 70)

missing = df_new.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

print("\n" + "=" * 70)
print("MISSING VALUE PERCENTAGE")
print("=" * 70)

missing_pct = (df_new.isna().mean() * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0])

print("\n" + "=" * 70)
print("NUMERIC SUMMARY OF IMPORTANT COLUMNS")
print("=" * 70)

important_numeric = [
    col for col in [
        "year", "month", "day", "cc_num", "mcc",
        "amount", "lat", "long", "city_pop",
        "income", "unix_time", "merch_lat", "merch_long",
        "is_foreign", "days_since_last_transaction",
        "transaction_amount_delta"
    ]
    if col in df_new.columns
]

print(df_new[important_numeric].describe().T)

TARGET DISTRIBUTIONS

--- is_fraud ---
Data type: int64
Missing: 0
is_fraud
0    4000
1     784
Name: count, dtype: int64

--- is_anomaly ---
Data type: int64
Missing: 0
is_anomaly
0    3784
1    1000
Name: count, dtype: int64

MISSING VALUES BY COLUMN
errors    4621
dtype: int64

MISSING VALUE PERCENTAGE
errors    96.592809
dtype: float64

NUMERIC SUMMARY OF IMPORTANT COLUMNS
                              count          mean           std           min  \
year                         4784.0  2.021286e+03  1.022721e+00  2.020000e+03   
month                        4784.0  6.119774e+00  3.388694e+00  1.000000e+00   
day                          4784.0  1.569105e+01  8.797384e+00  1.000000e+00   
cc_num                       4784.0  3.545239e+15  1.802775e+15  3.400258e+14   
mcc                          4784.0  5.732270e+03  1.222720e+03  3.000000e+03   
amount                       4784.0  1.440864e+02  3.036142e+02  3.000000e-02   
lat                          4784.0  2.931458e+01  2.

In [22]:
# ============================================================
# NEWER DATASET — COMPLETE FRAUD DETECTION PIPELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

import joblib
import os


# ============================================================
# 1. LOAD DATASET
# ============================================================

newer_file = (
    "../dataset/newer synthatic dataset/"
    "credit-card-fraud-xai-dataset-main/"
    "credit_card_fraud_sample.csv"
)

df_new = pd.read_csv(newer_file)

print("=" * 70)
print("NEWER DATASET")
print("=" * 70)

print("Shape:", df_new.shape)
print("Fraud cases:", df_new["is_fraud"].sum())
print("Legitimate cases:", (df_new["is_fraud"] == 0).sum())


# ============================================================
# 2. CREATE USEFUL TIME FEATURES
# ============================================================

df_new["trans_date_trans_time"] = pd.to_datetime(
    df_new["trans_date_trans_time"],
    errors="coerce"
)

df_new["transaction_hour"] = (
    df_new["trans_date_trans_time"].dt.hour
)

df_new["transaction_dayofweek"] = (
    df_new["trans_date_trans_time"].dt.dayofweek
)

# Drop the original timestamp after extracting useful features
df_new = df_new.drop(columns=["trans_date_trans_time"])


# ============================================================
# 3. DEFINE TARGET
# ============================================================

y_new = df_new["is_fraud"].astype(int)


# ============================================================
# 4. REMOVE TARGET / LEAKAGE / IDENTIFIER / PII COLUMNS
# ============================================================

columns_to_drop = [
    # Targets / labels
    "is_fraud",
    "is_anomaly",

    # Identifiers
    "cc_num",
    "user_id",
    "card_id",
    "trans_num",

    # Personal identifying information
    "first",
    "last",
    "street",
    "city",
    "zip",
    "dob",

    # Extremely sparse field
    "errors"
]

columns_to_drop = [
    col for col in columns_to_drop
    if col in df_new.columns
]

X_new = df_new.drop(columns=columns_to_drop)

print("\nFeatures after removing leakage/PII:")
print(X_new.columns.tolist())


# ============================================================
# 5. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

numeric_features = X_new.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_new.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


# ============================================================
# 6. TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_dev, X_test_new, y_dev, y_test_new = train_test_split(
    X_new,
    y_new,
    test_size=0.20,
    random_state=42,
    stratify=y_new
)

X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)

print(
    f"Train: {X_train_new.shape} | "
    f"Fraud: {y_train_new.sum()}"
)

print(
    f"Validation: {X_val_new.shape} | "
    f"Fraud: {y_val_new.sum()}"
)

print(
    f"Test: {X_test_new.shape} | "
    f"Fraud: {y_test_new.sum()}"
)


# ============================================================
# 7. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_features
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    )
])


# ============================================================
# 8. FIT PREPROCESSOR ONLY ON LEGITIMATE TRAINING DATA
# ============================================================

X_train_normal = X_train_new[
    y_train_new == 0
]

preprocessor.fit(X_train_normal)

X_train_normal_processed = (
    preprocessor.transform(X_train_normal)
)

X_val_processed = (
    preprocessor.transform(X_val_new)
)

X_test_processed = (
    preprocessor.transform(X_test_new)
)

print("\nPreprocessing completed! ✅")

print(
    "Processed training shape:",
    X_train_normal_processed.shape
)


# ============================================================
# 9. TRAIN ISOLATION FOREST
# ============================================================

new_model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

new_model.fit(X_train_normal_processed)

print("\nIsolation Forest trained! ✅")


# ============================================================
# 10. VALIDATION ANOMALY SCORES
# ============================================================

val_decision_scores_new = (
    new_model.decision_function(X_val_processed)
)

val_anomaly_scores_new = -val_decision_scores_new

print("\nValidation anomaly scores generated! ✅")

print(
    "Min:",
    val_anomaly_scores_new.min()
)

print(
    "Max:",
    val_anomaly_scores_new.max()
)

print(
    "Mean:",
    val_anomaly_scores_new.mean()
)


# ============================================================
# 11. VALIDATION THRESHOLD TUNING
# ============================================================

thresholds_new = np.linspace(
    val_anomaly_scores_new.min(),
    val_anomaly_scores_new.max(),
    200
)

threshold_results_new = []

for threshold in thresholds_new:

    y_val_pred_new = (
        val_anomaly_scores_new >= threshold
    ).astype(int)

    precision = precision_score(
        y_val_new,
        y_val_pred_new,
        zero_division=0
    )

    recall = recall_score(
        y_val_new,
        y_val_pred_new,
        zero_division=0
    )

    f1 = f1_score(
        y_val_new,
        y_val_pred_new,
        zero_division=0
    )

    threshold_results_new.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })


threshold_results_new = pd.DataFrame(
    threshold_results_new
)

best_row_new = threshold_results_new.loc[
    threshold_results_new["F1"].idxmax()
]

selected_threshold_new = (
    best_row_new["Threshold"]
)

print("\n" + "=" * 70)
print("BEST VALIDATION THRESHOLD")
print("=" * 70)

print(
    f"Threshold : {selected_threshold_new:.6f}"
)

print(
    f"Precision : {best_row_new['Precision']:.4f}"
)

print(
    f"Recall    : {best_row_new['Recall']:.4f}"
)

print(
    f"F1 Score  : {best_row_new['F1']:.4f}"
)


# ============================================================
# 12. FINAL TEST EVALUATION
# ============================================================

test_decision_scores_new = (
    new_model.decision_function(X_test_processed)
)

test_anomaly_scores_new = (
    -test_decision_scores_new
)

y_test_pred_new = (
    test_anomaly_scores_new >= selected_threshold_new
).astype(int)


# ============================================================
# 13. CALCULATE FINAL METRICS
# ============================================================

accuracy_new = accuracy_score(
    y_test_new,
    y_test_pred_new
)

precision_new = precision_score(
    y_test_new,
    y_test_pred_new,
    zero_division=0
)

recall_new = recall_score(
    y_test_new,
    y_test_pred_new,
    zero_division=0
)

f1_new = f1_score(
    y_test_new,
    y_test_pred_new,
    zero_division=0
)

roc_auc_new = roc_auc_score(
    y_test_new,
    test_anomaly_scores_new
)

pr_auc_new = average_precision_score(
    y_test_new,
    test_anomaly_scores_new
)

cm_new = confusion_matrix(
    y_test_new,
    y_test_pred_new
)


# ============================================================
# 14. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL NEWER DATASET TEST RESULTS")
print("=" * 70)

print(
    f"Accuracy  : {accuracy_new:.4f}"
)

print(
    f"Precision : {precision_new:.4f}"
)

print(
    f"Recall    : {recall_new:.4f}"
)

print(
    f"F1 Score  : {f1_new:.4f}"
)

print(
    f"ROC-AUC   : {roc_auc_new:.4f}"
)

print(
    f"PR-AUC    : {pr_auc_new:.4f}"
)

print("\nConfusion Matrix:")
print(cm_new)


# ============================================================
# 15. SAVE NEWER DATASET ARTIFACTS
# ============================================================

joblib.dump(
    new_model,
    "../models/newer_isolation_forest.pkl"
)

joblib.dump(
    preprocessor,
    "../models/newer_preprocessor.pkl"
)

joblib.dump(
    selected_threshold_new,
    "../models/newer_fraud_threshold.pkl"
)

print("\n" + "=" * 70)
print("NEWER DATASET ARTIFACTS SAVED")
print("=" * 70)

print("✓ newer_isolation_forest.pkl")
print("✓ newer_preprocessor.pkl")
print("✓ newer_fraud_threshold.pkl")

NEWER DATASET
Shape: (4784, 38)
Fraud cases: 784
Legitimate cases: 4000

Features after removing leakage/PII:
['year', 'month', 'day', 'time', 'card_network', 'payment_method', 'merchant', 'category', 'mcc', 'amount', 'gender', 'state', 'region', 'lat', 'long', 'city_pop', 'job', 'income', 'unix_time', 'merch_lat', 'merch_long', 'is_foreign', 'days_since_last_transaction', 'transaction_amount_delta', 'transaction_hour', 'transaction_dayofweek']

Numerical features:
['year', 'month', 'day', 'mcc', 'amount', 'lat', 'long', 'city_pop', 'income', 'unix_time', 'merch_lat', 'merch_long', 'is_foreign', 'days_since_last_transaction', 'transaction_amount_delta', 'transaction_hour', 'transaction_dayofweek']

Categorical features:
['time', 'card_network', 'payment_method', 'merchant', 'category', 'gender', 'state', 'region', 'job']

DATA SPLIT
Train: (2870, 26) | Fraud: 470
Validation: (957, 26) | Fraud: 157
Test: (957, 26) | Fraud: 157

Preprocessing completed! ✅
Processed training shape: (2400,

In [23]:
print("=" * 70)
print("FINAL NEWER DATASET TEST RESULTS")
print("=" * 70)

print(f"Threshold : {selected_threshold_new:.6f}")
print(f"Accuracy  : {accuracy_new:.4f}")
print(f"Precision : {precision_new:.4f}")
print(f"Recall    : {recall_new:.4f}")
print(f"F1 Score  : {f1_new:.4f}")
print(f"ROC-AUC   : {roc_auc_new:.4f}")
print(f"PR-AUC    : {pr_auc_new:.4f}")

print("\nConfusion Matrix:")
print(cm_new)

FINAL NEWER DATASET TEST RESULTS
Threshold : -0.009585
Accuracy  : 0.3877
Precision : 0.1784
Recall    : 0.7580
F1 Score  : 0.2888
ROC-AUC   : 0.5780
PR-AUC    : 0.2050

Confusion Matrix:
[[252 548]
 [ 38 119]]
